# Notebook 12: Knowledge Distillation

**Module:** 11 Production ML  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Understand teacher-student training
2. Derive distillation loss with temperature
3. Compress UNet++ teacher to smaller student
4. Connect to DINO self-distillation (Module 10)

## Knowledge Distillation

**Teacher:** Large accurate model (UNet++ SE-ResNet50)

**Student:** Smaller fast model (UNet++ MobileNet encoder)

$$L = \alpha L_{hard} + (1-\alpha) T^2 \cdot KL(\text{softmax}(z_s/T), \text{softmax}(z_t/T))$$

**Temperature T:** Softens probability distributions — transfers dark knowledge.

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, targets, T=4.0, alpha=0.5):
    hard = nn.CrossEntropyLoss()(student_logits, targets)
    soft = nn.KLDivLoss(reduction='batchmean')(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
    ) * (T * T)
    return alpha * hard + (1 - alpha) * soft

s = torch.randn(4, 10)
t = torch.randn(4, 10)
y = torch.randint(0, 10, (4,))
print(f'Distillation loss: {distillation_loss(s, t, y):.4f}')

---

## Summary

Distillation compresses teacher knowledge into deployable student models.

**Next:** [13_Docker_for_ML.ipynb](13_Docker_for_ML.ipynb)